# Export the full document-classification model to Hugging Face

This notebook loads the LoRA adapter from this GitHub project, merges it into `google/gemma-2-2b-it`, and uploads the resulting standalone model and tokenizer to a Hugging Face model repository.

## Requirements and cautions

- Use a Colab GPU with enough memory for the base model and merged weights. A T4 may work, but merging can require additional RAM/VRAM; use a high-memory runtime if available.
- Accept the [Gemma 2 2B terms](https://huggingface.co/google/gemma-2-2b-it) before downloading the gated base model.
- Add a Hugging Face token with **write** access to Colab Secrets as `HF_TOKEN`. It is only read at runtime and is not written into the notebook.
- Set the destination repository ID below. The repo is created **private by default**. Before making a Gemma-derived model public, check and follow the applicable Gemma terms and Hugging Face requirements.
- The adapter was trained for document format and administrative purpose classification. Merging it does not add training for other extracted metadata fields.


In [13]:
%pip -q install transformers peft accelerate huggingface_hub "torchao>=0.16.0"

## Configure source and destination

The adapter path is relative to the GitHub repository root. Change these settings if your repository, branch, or folder differs.

In [14]:
from pathlib import Path
import subprocess
import shutil

GITHUB_REPO_URL = "https://github.com/edla4eva/capstone.git"
GITHUB_BRANCH = "main"
ADAPTER_PATH = Path("gemma-instruction-adapter")
REPO_DIR = Path("/content/temp_repo")

HF_MODEL_REPO_ID = "edla4eva/gemma-admin-document-classifier"
MAKE_REPO_PRIVATE = False
BASE_MODEL_ID = "google/gemma-2-2b-it"

if "YOUR_USERNAME" in HF_MODEL_REPO_ID:
    raise ValueError("Set HF_MODEL_REPO_ID to your Hugging Face username and model repository name.")

# If the directory exists but is not a valid git repository or is empty, clean and clone it.
if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    print(f"Cleaning up incomplete directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    print(f"Cloning {GITHUB_REPO_URL} into {REPO_DIR}...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f"Repository already cloned at {REPO_DIR}")

# Adjust ADAPTER_DIR structure mapping based on actual repository structure
ADAPTER_DIR = REPO_DIR / ADAPTER_PATH
if not (ADAPTER_DIR / "adapter_model.safetensors").is_file():
    # Try alternate path if the first one doesn't exist
    alternate_path = REPO_DIR / "capstone" / ADAPTER_PATH
    if (alternate_path / "adapter_model.safetensors").is_file():
        ADAPTER_DIR = alternate_path
    else:
        raise FileNotFoundError(
            f"LoRA weights not found in {ADAPTER_DIR} or {alternate_path}. Check GITHUB_REPO_URL, GITHUB_BRANCH, and ADAPTER_PATH."
        )

print(f"Adapter verified at: {ADAPTER_DIR}")
print(f"Hugging Face destination: {HF_MODEL_REPO_ID} (private={MAKE_REPO_PRIVATE})")

Repository already cloned at /content/temp_repo
Adapter verified at: /content/temp_repo/gemma-instruction-adapter
Hugging Face destination: edla4eva/gemma-admin-document-classifier (private=False)


## Authenticate

Use a Hugging Face write token: it needs permission to download the gated base model and create/upload the destination model repository.

In [15]:
from huggingface_hub import HfApi

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_WRITE")
except (ImportError, KeyError, userdata.SecretNotFoundError):
    hf_token = None

if not hf_token:
    from getpass import getpass
    hf_token = getpass("Hugging Face write token (input hidden): ")
if not hf_token:
    raise ValueError("A Hugging Face token with write access is required.")

api = HfApi(token=hf_token)
api.create_repo(
    repo_id=HF_MODEL_REPO_ID,
    repo_type="model",
    private=MAKE_REPO_PRIVATE,
    exist_ok=True,
)
print("Destination repository is ready.")

Destination repository is ready.


## Load, merge, and upload

The model is loaded in FP16 to reduce memory use. `merge_and_unload()` folds the LoRA weights into the base-model weights so inference no longer needs PEFT or the separate adapter.

In [16]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU runtime before loading and merging the model.")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=hf_token)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token,
    low_cpu_mem_usage=True,
)
adapter_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
merged_model = adapter_model.merge_and_unload(safe_merge=True)
merged_model.eval()

print("LoRA adapter merged into the base model.")

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

W1122 22:56:28.828000 1478 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


LoRA adapter merged into the base model.


## Upload the standalone model and tokenizer

This uploads a complete model (larger than the adapter-only folder). Keep the Colab session active until both uploads finish.

In [18]:
merged_model.push_to_hub(
    HF_MODEL_REPO_ID,
    token=hf_token,
    private=False, # Explicitly set to False to ensure the repository remains public
    max_shard_size="2GB",
)
tokenizer.push_to_hub(
    HF_MODEL_REPO_ID,
    token=hf_token,
    private=False
)
print(f"Full model uploaded: https://huggingface.co/{HF_MODEL_REPO_ID}")

Writing model shards:   0%|          | 0/3 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0001-of-00003.safetensors:   1%|          | 16.0MB / 1.96GB            

  ...0002-of-00003.safetensors:   0%|          | 11.5kB / 2.00GB            

  ...0003-of-00003.safetensors:   0%|          |  524kB / 1.27GB            

... (3 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpnr5ityoq/tokenizer.json:  70%|######9   | 24.0MB / 34.4MB            

Validating                    :           |   0%            

Full model uploaded: https://huggingface.co/edla4eva/gemma-admin-document-classifier


## Verify by reloading the uploaded full model

This check confirms that the destination contains a complete model loadable without PEFT. It downloads the uploaded weights again.

In [19]:
del adapter_model, base_model, merged_model
torch.cuda.empty_cache()

check_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO_ID, token=hf_token)
check_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_REPO_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token,
)
print("Verified: the uploaded repository loads as a standalone model.")
del check_model, check_tokenizer
torch.cuda.empty_cache()
hf_token = None

config.json:   0%|          | 0.00/1.59k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/518 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/591 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Verified: the uploaded repository loads as a standalone model.


## Test Model Inference

Let's run a test inference to verify that the reloaded model correctly processes administrative documents using the fine-tuned instruction format.

In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Reload the tokenizer and model for the test
test_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO_ID, token=hf_token)
test_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_REPO_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token,
)

# Define a sample document prompt
sample_prompt = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\nClassify the following document content into its format and administrative purpose.\n\n"
    "### Input:\nMEMORANDUM\nTO: All Staff\nFROM: Administrative Board\nDATE: October 24, 2023\n"
    "SUBJECT: Annual Policy Review Updates and Compliance Guidelines\n\n"
    "### Response:\n"
)

# Run the inference
inputs = test_tokenizer(sample_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = test_model.generate(**inputs, max_new_tokens=50, temperature=0.1, do_sample=False)

response = test_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("--- Sample Prompt Input ---")
print(sample_prompt)
print("--- Model Classification Response ---")
print(response)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- Sample Prompt Input ---
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Classify the following document content into its format and administrative purpose.

### Input:
MEMORANDUM
TO: All Staff
FROM: Administrative Board
DATE: October 24, 2023
SUBJECT: Annual Policy Review Updates and Compliance Guidelines

### Response:

--- Model Classification Response ---
Format: Memorandum (memo)
Administrative Purpose: Notice of policy review and compliance guidelines
